Lets explore the datasets of RSNA and VinDr for pathology localization.

After review, only the VinDr datasets has bounding box labels. The RSNA dataset is not to be considered then.

This block should load some DICOM and show the bounding box with their markings.

The informations about the localized findings is in the finding_annotations.csv.

I'm gonna find all the annotations with the study_id of the locally downloaded data : 
- `0025a5dc99fd5c742026f0b2b030d3e9`
- `0028fb2c7f0b3a5cb9a80cb0e1cdbb91`
- `fd2fccc2650d1857a5669c28441300b2`
- `57674c2def92287668ed5bc51092fbef`



In [10]:
import os
from glob import glob

VINDR_PATH="/home/louis/Documents/personal/uni/session_7/apprentissage_auto/gif-7005_apprentissage_automatique/data/vindr"

STUDY_1 = "0025a5dc99fd5c742026f0b2b030d3e9"
IMAGE_STUDY_1 = glob(os.path.join(VINDR_PATH, "images", STUDY_1, "*"))

STUDY_2 = "0028fb2c7f0b3a5cb9a80cb0e1cdbb91"
IMAGE_STUDY_2 = glob(os.path.join(VINDR_PATH, "images", STUDY_2, "*"))

STUDY_3 = "fd2fccc2650d1857a5669c28441300b2"
IMAGE_STUDY_3 = glob(os.path.join(VINDR_PATH, "images", STUDY_3, "*"))

STUDY_4 = "57674c2def92287668ed5bc51092fbef"
IMAGE_STUDY_4 = glob(os.path.join(VINDR_PATH, "images", STUDY_4, "*"))


In [ ]:
%pip install pydicom
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

annotations = pd.read_csv(os.path.join(VINDR_PATH, "finding_annotations.csv"))

def show_mammogram(path, annotations):
    image_id = os.path.splitext(os.path.basename(path))[0]
    ds = pydicom.dcmread(path)
    img = ds.pixel_array

    rows = annotations[annotations["image_id"] == image_id]

    print("amount of findings : ", len(rows))
    print(rows)

    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(img, cmap="gray")

    for _, row in rows.iterrows():
        if "No Finding" in row["finding_categories"]:
            continue

        ax.add_patch(Rectangle(
            (row["xmin"], row["ymin"]),
            row["xmax"] - row["xmin"],
            row["ymax"] - row["ymin"],
            fill=False,
            edgecolor="red",
            linewidth=2,
        ))
        ax.text(
            row["xmin"],
            row["ymin"] - 20,
            row["finding_categories"],
            color="red",
            fontsize=8,
        )

    view = f"{rows.iloc[0]['laterality']} {rows.iloc[0]['view_position']}" if len(rows) else ""
    ax.set_title(f"{image_id} {view}")
    ax.axis("off")
    plt.show()



In [ ]:
# No findings
for path in sorted(IMAGE_STUDY_1):
    show_mammogram(path, annotations)

In [ ]:
# Findings BI-RADS 4
for path in sorted(IMAGE_STUDY_3):
    show_mammogram(path, annotations)

In [ ]:
# Findings BI-RADS 5
for path in sorted(IMAGE_STUDY_4):
    show_mammogram(path, annotations)

So I can vizualize the bounding boxes of the findings, now how to extract representations from models?

Loads the models -> make their last layer the prediction layer (what is the last layer for predicting bounding boxes?) -> Linear training for a fixed epochs -> metrics for bounding boxes -> there is also bounding box classification

It looks like it depends on the model -> we should conduct a research on how the four models does it : Mammo-FM, Mammo-Clip, Dinov2 and Dinov3

One bounding box per class? doesnt make any sense -> no multiple binding boxes

One class per training then -> looks like its the case

-> will need to launch multiple trainings per class

-> Mammo-FM : mAP (treats IoU of over 0.5 as True Positive) 

-> Mammo-Clip : mAP (treats IoU of over 0.5 as True Positive) aussi -> would be interesting

-> I would need to generalize the bounding box generation to the 4 models

-> I could all use RetinaNet for every model to generate bounding boxes

-> not all DICOMS are the same size

-> Ok yeah I need to attach a RetinaNet for bounding boxes

-> Lets start with the simple Mammo-FM

# Convert dicoms to png
```
python submodules/Mammo-FM/src/preprocessing/preprocess_image_to_png_vindr.py \
  --base_folder="/home/louis/Documents/personal/uni/session_7/apprentissage_auto/gif-7005_apprentissage_automatique/data/vindr"
```


In [ ]:
# Mammo-FM

## Download Mammo-FM
from huggingface_hub import hf_hub_download
path = hf_hub_download(
    repo_id="batmanLab/Mammo-FM",
    filename="Mammo-FM_BatmanlabTrained_CLIP.tar",
    local_dir="/home/louis/Documents/personal/uni/session_7/apprentissage_auto/gif-7005_apprentissage_automatique/data/models/mammo_fm",
)
print(path)

## Setup Mammo-FM with RetinaNet for bounding box extractions
import sys
from pathlib import Path\

import numpy as np
import torch
import pydicom
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

ROOT = Path("/home/louis/Documents/personal/uni/session_7/apprentissage_auto/gif-7005_apprentissage_automatique")
CODEBASE = ROOT / "submodules" / "Mammo-FM" / "src" / "codebase"
CKPT = ROOT / "data" / "models" / "mammo_fm" / "Mammo-FM_BatmanlabTrained_CLIP.tar"

sys.path.insert(0, str(CODEBASE))
from Detectors.retinanet.detector_model import RetinaNet_efficientnet



/home/louis/Documents/personal/uni/session_7/apprentissage_auto/gif-7005_apprentissage_automatique/data/models/mammo_fm/Mammo-FM_BatmanlabTrained_CLIP.tar
